<a href="https://colab.research.google.com/github/Wbunker/google-ai-tutorials/blob/main/ADK_Learning_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - MultiAgents! 🚀

Welcome back, Agent Architect! This notebook dives into the heart of the Google Agent Development Kit (ADK): orchestrating teams of specialized agents to tackle complex, multi-step problems that a single agent cannot handle alone.

By the end of this session, you will be an expert in advanced agentic workflows:

- **SequentialAgent**: You'll learn to chain agents together, creating pipelines where the output of one agent becomes the input for the next.

- **LoopAgent**: You'll build iterative systems where agents can plan, critique, and refine their work until a specific goal is met, making them "perfectionists."

- **ParallelAgent**: You'll unleash efficiency by running multiple agents simultaneously and then synthesizing their collective findings into a single, comprehensive answer.

- **The Router**: You will construct a "master" router agent that intelligently analyzes a user's request and delegates it to the correct agent or workflow.

Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```

  (\__/)
  (•ㅅ•)
  /づ  📚       Enjoy learning AI Agents :)


```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: [Google AI Studio](https://aistudio.google.com/app/apikey)

 -----------------------------------------------------------------------------


```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4

from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent, SequentialAgent, LoopAgent, ParallelAgent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search, ToolContext
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part

print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


### Configure Your API Key
To use Gemini models, you need an API key from [Google AI Studio](https://aistudio.google.com/app/apikey). This section securely collects your key and configures it for the ADK.


In [2]:
# --- API Key Configuration ---
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print("✅ API key entered manually.")


✅ API key loaded from Colab Secrets.


In [3]:
# --- Set Environment Variables for ADK ---

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f"✅ API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print("✅ Using Google AI Studio (not Vertex AI).")


✅ API key configured (starts with 'AQ.Ab8...')
✅ Using Google AI Studio (not Vertex AI).


In [4]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

---
## Part 1: Multi-Agent Mayhem - Sequential Workflows 🧠→🤖→🤖

You've mastered single agents and memory. Now for the most advanced topic: making agents **work together in a sequence**.

Some tasks are too complex for one agent. A user might ask, "Find me a great restaurant and then tell me how to get there." This requires two different skills: food recommendation and navigation.

We'll build a system that can handle this by:
1.  Creating a new `transportation_agent` 🚗.
2.  Teaching our `router_agent` 🧠 to recognize these special "combo" requests.
3.  Writing Python code (the "orchestrator") that runs the agents in a sequence, passing the output of the first agent to the second.

```
                    +---------------------+
                    |    User Query 🗣️     |
                    +----------+----------+
                               |
                               v
                    +---------------------+
                    |   Router Agent 🤖    |
                    |  (Classify Request) |
                    +----------+----------+
                               |
          +--------------------+----------------------+
          |                    |                      |
          v                    v                      v
  +----------------+   +--------------------+  +----------------------+
  |  foodie_agent  |   | weekend_guide_agent|  |  day_trip_agent      |
  |  🍣 Food Search |   | 🎉 Event Discovery |  | 🧳 Trip Planner       |
  +----------------+   +--------------------+  +----------------------+
          |
          v
  +----------------------------+            (if combo request)
  |  Restaurant Recommendation |---------------------------+
  |  ex: "Best sushi is at X"  |                           |
  +----------------------------+                           v
                                                        +-----------------------+
                                                        | transportation_agent  |
                                                        | 🚗 Get Directions      |
                                                        +-----------------------+
                                                        | Input: origin, place  |
                                                        | Output: directions    |
                                                        +-----------------------+

Final Output: 🍽️ Recommendation + 🚗 Route Info
```


In [5]:
# --- Agent Definitions for our Specialist Team ---
# --- Agent Definition ---

day_trip_agent = Agent(
    name="day_trip_agent",
    model="gemini-2.5-flash",
    description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
    instruction="""
    You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

    Your Mission:
    Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

    Guidelines:
    1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
    2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
    3. **Real-Time Focus**: Search for current operating hours and special events.
    4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

    RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
    """,
    tools=[google_search]
)

foodie_agent = Agent(
    name="foodie_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are an expert food critic. Your goal is to find the absolute best food, restaurants, or culinary experiences based on a user's request. When you recommend a place, state its name clearly. For example: 'The best sushi is at **Jin Sho**.'"
)

weekend_guide_agent = Agent(
    name="weekend_guide_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are a local events guide. Your task is to find interesting events, concerts, festivals, and activities happening on a specific weekend."
)

transportation_agent = Agent(
    name="transportation_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are a navigation assistant. Given a starting point and a destination, provide clear directions on how to get from the start to the end."
)

# --- The Brain of the Operation: The Router Agent ---
# We update the router's instructions to know about the new 'combo' task.
router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about food, restaurants, or eating.
    - 'weekend_guide_agent': For queries about events, concerts, or activities happening on a specific timeframe like a weekend.
    - 'day_trip_agent': A general planner for any other day trip requests.
    - 'find_and_navigate_combo': Use this for complex queries that ask to *first find a place* and *then get directions* to it.

    Only return the single, most appropriate option's name and nothing else.
    """
)

# We'll create a dictionary of all our individual worker agents
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent,
    "weekend_guide_agent": weekend_guide_agent,
    "transportation_agent": transportation_agent, # Add the new agent!
}

print("🤖 Agent team assembled for sequential workflows!")

🤖 Agent team assembled for sequential workflows!


In [6]:
# --- Let's Test the Sequential Workflow! ---
import re

async def run_sequential_app():
    queries = [
        "I want to eat the best sushi in Palo Alto.", # Should go to foodie_agent
        "Are there any cool outdoor concerts this weekend?", # Should go to weekend_guide_agent
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station." # Should trigger the COMBO
    ]

    for query in queries:
        print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("🧠 Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"🚦 Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        if chosen_route == 'find_and_navigate_combo':
            print("\n--- Starting Find and Navigate Combo Workflow ---")

            # STEP 2a: Run the foodie_agent first
            foodie_session = await session_service.create_session(app_name=foodie_agent.name, user_id=my_user_id)
            foodie_response = await run_agent_query(foodie_agent, query, foodie_session, my_user_id)

            # STEP 2b: Extract the destination from the first agent's response
            # (This is a simple regex, a more robust solution might use a structured output format)
            match = re.search(r'\*\*(.*?)\*\*', foodie_response)
            if not match:
                print("🚨 Could not determine the restaurant name from the response.")
                continue
            destination = match.group(1)
            print(f"💡 Extracted Destination: {destination}")

            # STEP 2c: Create a new query and run the transportation_agent
            directions_query = f"Give me directions to {destination} from the Palo Alto Caltrain station."
            print(f"\n🗣️ New Query for Transport Agent: '{directions_query}'")
            transport_session = await session_service.create_session(app_name=transportation_agent.name, user_id=my_user_id)
            await run_agent_query(transportation_agent, directions_query, transport_session, my_user_id)

            print("--- Combo Workflow Complete ---")

        elif chosen_route in worker_agents:
            # This is a simple, single-agent route
            worker_agent = worker_agents[chosen_route]
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
        else:
            print(f"🚨 Error: Router chose an unknown route: '{chosen_route}'")

await run_sequential_app()


🗣️ Processing New Query: 'I want to eat the best sushi in Palo Alto.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'c66c0f5f-8bdb-4183-8315-794a77227cab'...
🚦 Router has selected route: 'foodie_agent'

🚀 Running query for agent: 'foodie_agent' in session: '8daf2043-deba-4eed-bd38-091f876de899'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""For the absolute best sushi experience in Palo Alto, I recommend you visit **MJ Sushi**.

MJ Sushi is a well-established restaurant that has garnered a loyal customer base and high ratings on Google and Yelp. It is renowned for its fresh ingredients and impeccable service, ensuring a memorable dining experience. Many praise it for its delicious food, friendly service, and welcoming environment, with one reviewer explicitly stating it's the "Best sushi in Palo Alto!". Their diverse menu features a wide range of expertly crafted rolls, sashimi, nig

For the absolute best sushi experience in Palo Alto, I recommend you visit **MJ Sushi**.

MJ Sushi is a well-established restaurant that has garnered a loyal customer base and high ratings on Google and Yelp. It is renowned for its fresh ingredients and impeccable service, ensuring a memorable dining experience. Many praise it for its delicious food, friendly service, and welcoming environment, with one reviewer explicitly stating it's the "Best sushi in Palo Alto!". Their diverse menu features a wide range of expertly crafted rolls, sashimi, nigiri, and other Japanese dishes.

If you are looking for a more refined, high-end culinary journey, **Iki Omakase** offers a modern, sophisticated sushi omakase concept. Led by Chef Jiabo Li, Iki Omakase focuses on utilizing the finest globally and locally sourced ingredients, celebrating sushi as an art form with a unique Bay Area style rooted in Edomae sushi.

Another highly regarded option is **Jin Sho**, famously known as Steve Jobs' favorite restaurant. It's celebrated for its yellowtail jalapeño sashimi and spicy tuna rolls, offering great sushi, though portions can be small and prices higher. Jin Sho is described as a down-to-earth Japanese restaurant with a strong reputation for quality.

--------------------------------------------------


🗣️ Processing New Query: 'Are there any cool outdoor concerts this weekend?'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '8c6f4c42-332f-40a0-8794-43938112d400'...
🚦 Router has selected route: 'weekend_guide_agent'

🚀 Running query for agent: 'weekend_guide_agent' in session: 'ed3f4f54-63a7-4885-8452-d798aa1f63bf'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""I can help you find some cool outdoor concerts! To give you the best recommendations, could you please tell me:

1.  What is your general **location** (city and state/region)?
2.  Are you looking for concerts for **next weekend (July 11th-12th, 2026)**, or did you mean a different weekend?"""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata() partial=None turn_complete=None turn_complete_reason=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None e

I can help you find some cool outdoor concerts! To give you the best recommendations, could you please tell me:

1.  What is your general **location** (city and state/region)?
2.  Are you looking for concerts for **next weekend (July 11th-12th, 2026)**, or did you mean a different weekend?

--------------------------------------------------


🗣️ Processing New Query: 'Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '089906c9-904d-4fc9-aadb-f37c1ef768d4'...
🚦 Router has selected route: 'find_and_navigate_combo'

--- Starting Find and Navigate Combo Workflow ---

🚀 Running query for agent: 'foodie_agent' in session: 'dc26bbc2-05fb-4ac1-9ab7-032620c5271c'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""The finest sushi experience in Palo Alto awaits you at **Jin Sho**. Renowned for its elegant and traditional ambiance, Jin Sho offers a selection of fresh and delicious dishes, including highly-praised sashimi and an Omakase option. Their skilled sushi chefs, some with experience from Nobu in New York, are passionate about their craft, ensuring a high-quality dining experience. You

The finest sushi experience in Palo Alto awaits you at **Jin Sho**. Renowned for its elegant and traditional ambiance, Jin Sho offers a selection of fresh and delicious dishes, including highly-praised sashimi and an Omakase option. Their skilled sushi chefs, some with experience from Nobu in New York, are passionate about their craft, ensuring a high-quality dining experience. You can find Jin Sho at **454 California Avenue, Palo Alto, California 94306**.

To get to Jin Sho from the Caltrain station, I recommend arriving at the **California Avenue Caltrain Station**. This station is located directly on California Avenue (at 101 California Ave), making it an incredibly convenient starting point for your short walk to Jin Sho.

**Walking directions from California Avenue Caltrain Station to Jin Sho:**

1.  Exit the California Avenue Caltrain Station.
2.  Walk southeast on California Avenue towards El Camino Real. Jin Sho is located at 454 California Avenue, which will be a short walk from the station along the same street.

If you arrive at the main **Palo Alto Caltrain Station (University Avenue)**, located at 95 University Ave, please be aware that Jin Sho is further away. From this station, a taxi or rideshare service would be the most practical option, as the walk to California Avenue can take approximately 30-38 minutes.

--------------------------------------------------

💡 Extracted Destination: Jin Sho

🗣️ New Query for Transport Agent: 'Give me directions to Jin Sho from the Palo Alto Caltrain station.'

🚀 Running query for agent: 'transportation_agent' in session: 'c4b5edf8-cf2b-429b-9285-18468f8821e0'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""To get to Jin Sho, located at 454 California Avenue in Palo Alto, from the Palo Alto Caltrain Station (95 University Avenue), you have several transportation options.

**Important Note:** There are two Caltrain stations in Palo Alto: the main Palo Alto Caltrain Station on University Avenue, and the California Avenue Caltrain Station. Jin Sho is located on California Avenue, making the California Avenue Caltrain Station a much closer option if that is where you arrive. If you are starting from the main Palo Alto Caltrain Station on University Avenue, follow the directions below.

### From Palo Alto Caltrain S

To get to Jin Sho, located at 454 California Avenue in Palo Alto, from the Palo Alto Caltrain Station (95 University Avenue), you have several transportation options.

**Important Note:** There are two Caltrain stations in Palo Alto: the main Palo Alto Caltrain Station on University Avenue, and the California Avenue Caltrain Station. Jin Sho is located on California Avenue, making the California Avenue Caltrain Station a much closer option if that is where you arrive. If you are starting from the main Palo Alto Caltrain Station on University Avenue, follow the directions below.

### From Palo Alto Caltrain Station (University Avenue) to Jin Sho (454 California Avenue)

The distance between the main Palo Alto Caltrain Station and Jin Sho is approximately 2.5 to 3 miles.

**1. By Public Transportation (Bus and Walk - Recommended)**

This is often the most convenient public transport option as it combines a bus ride with a short walk.

*   **Walk to a nearby bus stop:** From the Palo Alto Caltrain Station, walk to a bus stop for a VTA bus heading towards California Avenue. The VTA Line 22 and Rapid 522 serve the Palo Alto area.
*   **Take the bus:** Board a VTA bus (such as the 22 or Rapid 522) traveling south on El Camino Real.
*   **Exit at California Avenue:** Get off the bus near the intersection of El Camino Real and California Avenue.
*   **Walk to Jin Sho:** From there, Jin Sho at 454 California Avenue is a short walk west on California Avenue.

**2. By Taxi or Rideshare Service**

Taking a taxi or using a rideshare service (like Uber or Lyft) will be the quickest way to get directly to Jin Sho. The ride will typically take around 5-10 minutes depending on traffic.

*   Exit the Palo Alto Caltrain Station and locate the designated pick-up area for taxis or rideshare services.
*   Provide the driver with the address: 454 California Avenue, Palo Alto.

**3. By Car (Driving)**

If you have access to a car, the drive is straightforward:

*   **Exit the Caltrain Station:** From 95 University Avenue, head southeast on University Avenue.
*   **Turn right onto El Camino Real (CA-82 S):** Drive south on El Camino Real for approximately 2-2.5 miles.
*   **Turn right onto California Avenue:** Jin Sho will be on your right-hand side shortly after turning onto California Avenue.

**4. Walking**

Walking from the main Palo Alto Caltrain Station to Jin Sho would be a long walk (approximately 45-60 minutes) and is generally not recommended unless you prefer a long stroll.

*   From the station, head south on Alma Street or parallel streets.
*   You will eventually need to make your way west to California Avenue.

### If you are at the California Avenue Caltrain Station

If you arrived at the California Avenue Caltrain Station (101 California Avenue), Jin Sho is just a short walk away.

*   Exit the California Avenue Caltrain Station.
*   Walk west on California Avenue towards 454 California Avenue. Jin Sho will be on your left. This walk should take approximately 2-5 minutes.

--------------------------------------------------

--- Combo Workflow Complete ---


---
### Part 2 (The ADK Way): Multi-Agent Mayhem with `SequentialAgent` 🧠→⛓️→🤖

You've seen how to manually link agents together with custom Python code. It works, but it can get complicated. Now, let's refactor our workflow to use a powerful, built-in ADK feature designed specifically for this: the **`SequentialAgent`**.

The `SequentialAgent` is a *workflow agent*. It's not powered by an LLM itself; instead, its only job is to execute a list of other agents in a strict, predefined order.

The real magic ✨ is how it passes information. The ADK uses a shared `state` dictionary that each agent in the sequence can read from and write to.

**Our New Workflow:**

1.  **Foodie Agent**: Finds the restaurant and saves the name to `state['destination']`.
2.  **Transportation Agent**: Automatically reads `state['destination']` and uses it to find directions.

This means we no longer need custom Python code to extract text or build new queries! The ADK handles the plumbing for us.

```
+-------------------------------+
|  find_and_navigate_agent 🧭   |
| SequentialAgent:             |
| 1. Find destination          |
| 2. Get directions            |
+---------------+---------------+
                |
     +----------+----------+
     |                     |
     v                     v
+----------------+   +------------------------+
| foodie_agent 🍣 |   | transportation_agent 🚗 |
| Finds place     |   | Uses {destination}     |
| Output: 'Jin Sho'|   | Output: Directions     |
+----------------+   +------------------------+

Final Output: 🍣 Restaurant + 🚗 Route
```

In [7]:
# --- Agent Definitions for our Specialist Team (Refactored for Sequential Workflow) ---

# ✨ CHANGE 1: We tell foodie_agent to save its output to the shared state.
# Note the new `output_key` and the more specific instruction.
foodie_agent = Agent(
    name="foodie_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are an expert food critic. Your goal is to find the best restaurant based on a user's request.

    When you recommend a place, you must output *only* the name of the establishment and nothing else.
    For example, if the best sushi is at 'Jin Sho', you should output only: Jin Sho
    """,
    output_key="destination"  # ADK will save the agent's final response to state['destination']
)

# ✨ CHANGE 2: We tell transportation_agent to read from the shared state.
# The `{destination}` placeholder is automatically filled by the ADK from the state.
transportation_agent = Agent(
    name="transportation_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are a navigation assistant. Given a destination, provide clear directions.
    The user wants to go to: {destination}.

    Analyze the user's full original query to find their starting point.
    Then, provide clear directions from that starting point to {destination}.
    """,
)

# ✨ CHANGE 3: Define the SequentialAgent to manage the workflow.
# This agent will run foodie_agent, then transportation_agent, in that exact order.
find_and_navigate_agent = SequentialAgent(
    name="find_and_navigate_agent",
    sub_agents=[foodie_agent, transportation_agent],
    description="A workflow that first finds a location and then provides directions to it."
)

weekend_guide_agent = Agent(
    name="weekend_guide_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are a local events guide. Your task is to find interesting events, concerts, festivals, and activities happening on a specific weekend."
)

# --- The Brain of the Operation: The Router Agent ---

# We update the router to know about our new, powerful SequentialAgent.
router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about food, restaurants, or eating.
    - 'weekend_guide_agent': For queries about events, concerts, or activities happening on a specific timeframe like a weekend.
    - 'day_trip_agent': A general planner for any other day trip requests.
    - 'find_and_navigate_agent': Use this for complex queries that ask to *first find a place* and *then get directions* to it.

    Only return the single, most appropriate option's name and nothing else.
    """
)

# We create a dictionary of all our executable agents for easy lookup.
# This now includes our powerful new workflow agent!
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent,
    "weekend_guide_agent": weekend_guide_agent,
    "find_and_navigate_agent": find_and_navigate_agent, # Add the new sequential agent
}

print("🤖 Agent team assembled with a SequentialAgent workflow!")

🤖 Agent team assembled with a SequentialAgent workflow!


/tmp/ipykernel_803/57158224.py:33: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  find_and_navigate_agent = SequentialAgent(


In [8]:
# --- Let's Test the Streamlined Workflow! ---

async def run_sequential_app():
    queries = [
        "I want to eat the best sushi in Palo Alto.", # Should go to foodie_agent
        "Are there any cool outdoor concerts this weekend?", # Should go to weekend_guide_agent
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station." # Should trigger the SequentialAgent
    ]

    for query in queries:
        print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("🧠 Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"🚦 Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        # This logic is now much simpler! The SequentialAgent is treated just like any other worker.
        if chosen_route in worker_agents:
            worker_agent = worker_agents[chosen_route]
            print(f"--- Handing off to {worker_agent.name} ---")
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
            print(f"--- {worker_agent.name} Complete ---")
        else:
            print(f"🚨 Error: Router chose an unknown route: '{chosen_route}'")

await run_sequential_app()


🗣️ Processing New Query: 'I want to eat the best sushi in Palo Alto.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '3c5b3472-9c20-4eaf-bdb1-1df1e1cb0a52'...
🚦 Router has selected route: 'foodie_agent'
--- Handing off to foodie_agent ---

🚀 Running query for agent: 'foodie_agent' in session: 'd71184f7-c8d7-420a-aab0-048edce81896'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='Jin Sho'
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webki

Jin Sho

--------------------------------------------------

--- foodie_agent Complete ---

🗣️ Processing New Query: 'Are there any cool outdoor concerts this weekend?'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'add69eb3-9bee-4556-a3b4-172beeac05d1'...
🚦 Router has selected route: 'weekend_guide_agent'
--- Handing off to weekend_guide_agent ---

🚀 Running query for agent: 'weekend_guide_agent' in session: 'c3389e7e-32a8-439f-8a0f-21cfeb0b673c'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""This upcoming weekend, July 11-12, 2026, offers a variety of exciting outdoor concerts and music festivals across different locations! Here are some highlights:

**Music Festivals:**

*   **Hodag Country Festival (Rhinelander, Wisconsin)**: This festival runs from July 9-12, with a strong lineup on Saturday, July 11th, featuring Old Dominion at 9:00 PM, LeAnn Rimes at 6:30 PM, Niko Moon at 4:30 PM, Mark

This upcoming weekend, July 11-12, 2026, offers a variety of exciting outdoor concerts and music festivals across different locations! Here are some highlights:

**Music Festivals:**

*   **Hodag Country Festival (Rhinelander, Wisconsin)**: This festival runs from July 9-12, with a strong lineup on Saturday, July 11th, featuring Old Dominion at 9:00 PM, LeAnn Rimes at 6:30 PM, Niko Moon at 4:30 PM, Mark Chesnutt at 2:30 PM, and Tyler Nance at 1:00 PM. On Sunday, July 12th, you can catch Nate Smith at 7:00 PM, ERNEST at 5:00 PM, Steve Earle with Reckless Kelly at 3:15 PM, Preston Cooper at 1:30 PM, and Lanie Gardner at 12:00 PM.
*   **Ultra Europe (Europe)**: For fans of electronic dance music, Ultra Europe is a three-day event happening from July 10-12, showcasing top EDM DJs.
*   **Blessing in Disguise Music Festival (Spokane, Washington)**: This immersive music festival is scheduled for July 11, 2026, offering a diverse lineup of music, food, art, and community events.
*   **America's Mountain Festival (Woodland Park, Colorado)**: Returning in 2026, this festival celebrates with a full day of live country and Americana music, featuring artists like Aaron Watson, Jenna Paulette, Tyce Delk, Walker Montgomery, and Matt Skinner Band.
*   **Pendleton Whisky Music Fest (Pendleton, Oregon)**: While the full lineup is still to be announced, this festival promises a great party on Saturday, July 11th.

**Outdoor Concerts and Series:**

*   **Atlanta, Georgia Area**: The Atlanta area has numerous free outdoor concert series on July 11th, including:
    *   Peachtree Corners Summer Concert Series (Peachtree Corners Town Green)
    *   Home by Dark outdoor concert series (Brooke Street Park, Alpharetta)
    *   Woodstock Summer Concert Series (Cherokee Amphitheatre, Woodstock)
    *   Avalon Nights Live (Avalon)
    *   Groovin' on the Square outdoor concert series (Colony Square)
    *   Rock the Block: Live & Loud Fridays (Duluth Town Green)
    *   Concerts by the Springs featuring Radio 80s Band (Heritage Green in Sandy Springs)
    *   Norcross Summer Concerts (Thrasher Park, Norcross)
    *   Glover Park Concert Series with Seven Bridges, an Eagles tribute band (Marietta Square)
    *   Cumming City Center concert series (Cumming City Center)
*   **San Diego, California**: SeaWorld San Diego's Summer Concerts will feature E-40 at the Bayside Amphitheater on Saturday, July 11th, at 6:00 p.m.
*   **Atlantic City, New Jersey**: On Sunday, July 12th, Boardwalk Hall is hosting a free summer concert starting at 4:00 PM, featuring international recording artists Tavares, Russell Thompkins Jr. & The New Stylistics, Wil Hart of The Original Delfonics, Billy Brown of Ray, Goodman & Brown, Blue Magic, Black Ivory, Allure, and Dennis Taylor.
*   **Somerville, New Jersey**: Downtown Somerville's 2026 Summer Stage Concert Series presents the Mason Band on July 11th from 6:00 pm to 9:00 pm on Division Street.
*   **Carnation, Washington (near Seattle)**: Remlinger Concerts at Remlinger Farms will feature Head and the Heart on July 10th and 11th at 6:00 PM.
*   **Troutdale, Oregon (near Portland)**: Edgefield Concerts on the Lawn will host Rainbow Kitten Surprise and Spacey Jane on Sunday, July 12th, with doors opening at 5:00 PM and the show starting at 6:30 PM.
*   **Highland Park, Illinois (near Chicago)**: The Ravinia Festival has Billy Idol with special guest Tom Hamilton Band performing at the Hunter Pavilion on Sunday, July 12th. On the same day, you can experience "Vienna's Golden Century" chamber music at the Bennett Gordon Hall.

To give you the most relevant information, could you please share your general location or the area you're interested in?

--------------------------------------------------

--- weekend_guide_agent Complete ---

🗣️ Processing New Query: 'Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '032ad11d-7c60-4830-b239-3a897ba0a17e'...
🚦 Router has selected route: 'find_and_navigate_agent'
--- Handing off to find_and_navigate_agent ---

🚀 Running query for agent: 'find_and_navigate_agent' in session: '17effc97-8bb7-47ed-8732-87fe353660f0'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Jin Sho

To get to Jin Sho from the California Avenue Caltrain station, exit the station and walk west on California Avenue for about 0.2 miles. Jin Sho will be on your left at 454 California Avenue."""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(


Based on recommendations and reviews, Jin Sho is considered one of the best sushi restaurants in Palo Alto. It was notably a favorite of Steve Jobs. Reviewers praise its fresh sushi, high-quality ingredients, and skilled chefs.

To get to Jin Sho from the California Avenue Caltrain station, exit the station and walk west on California Avenue for about 0.2 miles. Jin Sho will be on your left at 454 California Avenue.

--------------------------------------------------

--- find_and_navigate_agent Complete ---


### Running the Streamlined App

Notice how much simpler the code below is. There is no longer a special `if chosen_route == 'find_and_navigate_combo':` block with custom logic.

The `find_and_navigate_agent` is now a self-contained unit. We can treat it just like any other agent, hand it the query, and trust the `SequentialAgent` to handle all the internal steps. This makes our main application code much cleaner and easier to read.

---
## Iterative Ideas with `LoopAgent` 🧠→🔁→🤖

Sometimes a task isn't a straight line; it's a loop of refinement. A user might ask for a plan, but have constraints that require checking and re-planning. For this, the ADK provides the **`LoopAgent`**.

The `LoopAgent` executes a sequence of sub-agents repeatedly until a condition is met. This is perfect for workflows involving trial and error, like planning a trip with a tight schedule.

**Our New Workflow: The Perfectionist Planner**

1. **Planner Agent**: Proposes an itinerary (e.g., a museum and a restaurant).
2. **Critic Agent**: Checks the plan against a constraint (e.g., "Is the travel time between these two places less than 30 minutes?").
3. **Refiner Agent**: If the critic finds a problem, this agent takes the feedback and creates a new, improved plan. If the critic is happy, it calls a special `exit_loop` tool to stop the process.

The `LoopAgent` manages this cycle, ensuring we don't get stuck in an infinite loop by setting a `max_iterations` limit.

```
+-------------------------------+
|  iterative_planner_agent 🔁   |
| SequentialAgent:              |
| 1. Propose Plan               |
| 2. Refine in loop (≤ 3 times) |
+---------------+---------------+
                |
     +----------+----------+
     |                     |
     v                     v
+----------------+   +-----------------------+
| planner_agent  |   | refinement_loop 🔁     |
| Propose plan   |   | LoopAgent             |
| e.g., Activity +  | 1. Critic (time check) |
| Restaurant       | 2. Refiner (fix/exit)   |
+----------------+   +-----------------------+

Uses shared state: {current_plan}, {criticism}
Exits when: "Plan is feasible..."

```

In [10]:
# --- Agent Definitions for an Iterative Workflow ---

# A tool to signal that the loop should terminate
COMPLETION_PHRASE = "The plan is feasible and meets all constraints."
def exit_loop(tool_context: ToolContext):
  """Call this function ONLY when the plan is approved, signaling the loop should end."""
  print(f"  [Tool Call] exit_loop triggered by {tool_context.agent_name}")
  tool_context.actions.escalate = True
  return {}

# Agent 1: Proposes an initial plan
planner_agent = Agent(
    name="planner_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are a trip planner. Based on the user's request, propose a single activity and a single restaurant. Output only the names, like: 'Activity: Exploratorium, Restaurant: La Mar'.",
    output_key="current_plan"
)

# Agent 2 (in loop): Critiques the plan
critic_agent = Agent(
    name="critic_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction=f"""You are a logistics expert. Your job is to critique a travel plan. The user has a strict constraint: total travel time must be short.
    Current Plan: {{current_plan}}
    Use your tools to check the travel time between the two locations.
    IF the travel time is over 45 minutes, provide a critique, like: 'This plan is inefficient. Find a restaurant closer to the activity.'
    ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'""",
    output_key="criticism"
)

# Agent 3 (in loop): Refines the plan or exits
refiner_agent = Agent(
    name="refiner_agent", model="gemini-2.5-flash", tools=[google_search, exit_loop],
    instruction=f"""You are a trip planner, refining a plan based on criticism.
    Original Request: {{session.query}}
    Critique: {{criticism}}
    IF the critique is '{COMPLETION_PHRASE}', you MUST call the 'exit_loop' tool.
    ELSE, generate a NEW plan that addresses the critique. Output only the new plan names, like: 'Activity: de Young Museum, Restaurant: Nopa'.""",
    output_key="current_plan"
)

# ✨ The LoopAgent orchestrates the critique-refine cycle ✨
refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=3
)

# ✨ The SequentialAgent puts it all together ✨
iterative_planner_agent = SequentialAgent(
    name="iterative_planner_agent",
    sub_agents=[planner_agent, refinement_loop],
    description="A workflow that iteratively plans and refines a trip to meet constraints."
)

print("🤖 Agent team updated with an iterative LoopAgent workflow!")

🤖 Agent team updated with an iterative LoopAgent workflow!


/tmp/ipykernel_803/3500354978.py:41: DeprecationWarning: LoopAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  refinement_loop = LoopAgent(
/tmp/ipykernel_803/3500354978.py:48: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  iterative_planner_agent = SequentialAgent(


---
## Parallel Power with `ParallelAgent` 🧠→⚡️→🤖🤖🤖

What if a user wants to find multiple, unrelated things at once? "Find me a museum, a concert, AND a restaurant." Running these searches one by one is slow and inefficient.

Enter the **`ParallelAgent`**. This workflow agent executes a list of sub-agents *concurrently*, dramatically speeding up tasks that can be performed independently.

**Our New Workflow: The Multi-Researcher**

1.  **Parallel Agent**: Simultaneously runs three specialist agents:
    - `MuseumFinderAgent`: Finds a museum.
    - `ConcertFinderAgent`: Finds a concert.
    - `FoodieAgent`: Finds a restaurant.
2.  **Synthesis Agent**: Once all three parallel searches are complete, this final agent gathers the results (which were saved to the shared `state`) and formats them into a single, neat summary for the user.

This pattern lets us get a lot of work done, fast! 🚀

```
+-------------------------------+
|  parallel_planner_agent ⚡     |
| SequentialAgent:              |
| 1. Run parallel research      |
| 2. Synthesize results         |
+---------------+---------------+
                |
     +----------+----------------------+
     |                                 |
     v                                 v
+-------------------------+       +-----------------------------+
| parallel_research_agent ⚡   |   | synthesis_agent 📋          |
| ParallelAgent:              |   | Combine results            |
| - museum_finder_agent 🖼️     |   | Output: Bulleted summary   |
| - concert_finder_agent 🎵    |   +-----------------------------+
| - restaurant_finder_agent 🍽️ |
+-------------------------+

Final Output:
• Museum: XYZ  
• Concert: Artist at Venue  
• Restaurant: ABC
```

In [11]:
# --- Agent Definitions for a Parallel Workflow ---

# Specialist Agent 1
museum_finder_agent = Agent(
    name="museum_finder_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are a museum expert. Find the best museum based on the user's query. Output only the museum's name.",
    output_key="museum_result"
)

# Specialist Agent 2
concert_finder_agent = Agent(
    name="concert_finder_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are an events guide. Find a concert based on the user's query. Output only the concert name and artist.",
    output_key="concert_result"
)

# We can reuse our foodie_agent for the third parallel task!
# Just need to give it a new output_key for this workflow.
# restaurant_finder_agent = foodie_agent.copy(update={"output_key": "restaurant_result"})
restaurant_finder_agent = Agent(
    name="restaurant_finder_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are an expert food critic. Your goal is to find the best restaurant based on a user's request.

    When you recommend a place, you must output *only* the name of the establishment.
    For example, if the best sushi is at 'Jin Sho', you should output only: Jin Sho
    """,
    output_key="restaurant_result" # Set the correct output key for this workflow
)


# ✨ The ParallelAgent runs all three specialists at once ✨
parallel_research_agent = ParallelAgent(
    name="parallel_research_agent",
    sub_agents=[museum_finder_agent, concert_finder_agent, restaurant_finder_agent]
)

# Agent to synthesize the parallel results
synthesis_agent = Agent(
    name="synthesis_agent", model="gemini-2.5-flash",
    instruction="""You are a helpful assistant. Combine the following research results into a clear, bulleted list for the user.
    - Museum: {museum_result}
    - Concert: {concert_result}
    - Restaurant: {restaurant_result}
    """
)

# ✨ The SequentialAgent runs the parallel search, then the synthesis ✨
parallel_planner_agent = SequentialAgent(
    name="parallel_planner_agent",
    sub_agents=[parallel_research_agent, synthesis_agent],
    description="A workflow that finds multiple things in parallel and then summarizes the results."
)

print("🤖 Agent team supercharged with a ParallelAgent workflow!")

🤖 Agent team supercharged with a ParallelAgent workflow!


/tmp/ipykernel_803/983560783.py:34: DeprecationWarning: ParallelAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  parallel_research_agent = ParallelAgent(
/tmp/ipykernel_803/983560783.py:50: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  parallel_planner_agent = SequentialAgent(


---
### Final Step: Updating the Router and Running the App

Now we just have one last thing to do: make our `router_agent` aware of these powerful new workflows! We'll add `iterative_planner_agent` and `parallel_planner_agent` to its list of available options.

Then we can run our app with new queries designed to trigger these advanced, multi-agent workflows.

```
                    +---------------------+
                    |    User Query 🗣️     |
                    +----------+----------+
                               |
                               v
                    +---------------------+
                    |   Router Agent 🤖    |
                    |  (Classify Request) |
                    +----------+----------+
                               |
      +-----------+-----------+-----------+-----------+------------+
      |           |           |           |           |            |
      v           v           v           v           v            v
+-------------+  +------------------+  +------------------+  +------------------+  +-----------------+
| foodie_agent|  | find_and_navigate|  | iterative_planner|  | parallel_planner |  | day_trip_agent  |
| 🍣 Food Only |  | 🧭 Seq Workflow   |  | 🔁 Loop Workflow  |  | ⚡ Parallel Tasks |  | 🧳 Basic Plan     |
+-------------+  +------------------+  +------------------+  +------------------+  +-----------------+
```

In [ ]:
# --- The ULTIMATE Router Agent --- #

router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a master request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about finding a single food place.
    - 'find_and_navigate_agent': For queries that ask to *first find a place* and *then get directions* to it.
    - 'iterative_planner_agent': For planning a trip with a specific constraint that needs checking, like travel time.
    - 'parallel_planner_agent': For queries that ask to find multiple, independent things at once (e.g., a museum AND a concert AND a restaurant).
    - 'day_trip_agent': A general planner for any other simple day trip requests.

    Only return the single, most appropriate option's name and nothing else.
    """
)

# The master dictionary of all our executable agents and workflows
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent, # For simple food queries
    "find_and_navigate_agent": find_and_navigate_agent, # Sequential
    "iterative_planner_agent": iterative_planner_agent, # Loop
    "parallel_planner_agent": parallel_planner_agent,   # Parallel
}

# --- Let's Test Everything! ---

async def run_fully_loaded_app():
    queries = [
        # Test Case 1: Simple Sequential Flow
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.",

        # Test Case 2: Iterative Loop Flow
        "Plan me a day in San Francisco with a museum and a nice dinner, but make sure the travel time between them is very short.",

        # Test Case 3: Parallel Flow
        "Help me plan a trip to SF. I need one museum, one concert, and one great restaurant."
    ]

    for query in queries:
        print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("🧠 Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"🚦 Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        if chosen_route in worker_agents:
            worker_agent = worker_agents[chosen_route]
            print(f"--- Handing off to {worker_agent.name} ---")
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
            print(f"--- {worker_agent.name} Complete ---")
        else:
            print(f"🚨 Error: Router chose an unknown route: '{chosen_route}'")

await run_fully_loaded_app()


🗣️ Processing New Query: 'Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '4e909eca-d0a4-493e-9053-84d2df0736de'...
🚦 Router has selected route: 'An error occurred: name Content is not defined'
🚨 Error: Router chose an unknown route: 'An error occurred: name Content is not defined'

🗣️ Processing New Query: 'Plan me a day in San Francisco with a museum and a nice dinner, but make sure the travel time between them is very short.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '86586315-bddb-479a-90aa-09ae5afd09bd'...
🚦 Router has selected route: 'An error occurred: name Content is not defined'
🚨 Error: Router chose an unknown route: 'An error occurred: name Content is not defined'

🗣️ Processing New Query: 'Help me plan a trip to SF. I need one museum, one concert, and one great r

---
## 🎉 Congratulations! 🎉

You've completed the Enhanced ADK Adventure! You have successfully. Let's review the advanced orchestration patterns you've successfully implemented:

- **The Router Pattern**: You built a master router agent capable of analyzing user intent and delegating tasks to the appropriate specialist agent or workflow.

- **Sequential Workflows**: Using SequentialAgent, you elegantly chained agents together, creating clean, readable code for multi-step tasks without manual data handling.

- **Iterative Refinement**: You constructed a sophisticated feedback loop with LoopAgent, enabling your agents to plan, self-critique, and improve their output until it met specific constraints.

- **Parallel Power**: You maximized speed and efficiency by using ParallelAgent to run multiple research tasks concurrently, later synthesizing the results into a unified response.


```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
